# Regresión con zarigüeyas

## Pregunta de investigación
**¿Podemos usar la longitud total para predecir la longitud de la cabeza de una zarigüeya?**

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 1. Carga de datos

Se asume que el archivo `possum.csv` está en la misma carpeta de este notebook.  

In [ ]:
source_used = "possum.csv"
df = pd.read_csv(source_used)

print("Fuente utilizada:", source_used)
print("Dimensiones iniciales:", df.shape)
df.head()

## 2 Inspección inicial

### 2.1 Nombres de las columnas

In [ ]:
print("Columnas originales:")
print(list(df.columns))

### 2.2 Tipo de datos

In [ ]:
print("\nTipos de datos:")
print(df.dtypes)

### 2.3 Valores nulos

In [ ]:
print("\nValores faltantes por columna:")
print(df.isna().sum())

## 3. Preparación de datos (curado)

### 3.1 Selección de entradas y salidas

In [ ]:
input_features = ["totlngth"]
target_feature = "hdlngth"

print("\nVariables del modelo:")
print(" - Características de entrada:", input_features)
print(" - Variable de salida        :", target_feature)

Conservamos solo las columnas necesarias y eliminamos filas con valores faltantes

In [ ]:
df = df.dropna(subset=input_features + [target_feature]).copy()

print("\nDimensiones del conjunto de trabajo:", df.shape)
df[input_features + [target_feature]].head()

### 3.2 Estadística descriptiva

In [ ]:
display(df[input_features + [target_feature]].describe().T)

corr = df[input_features[0]].corr(df[target_feature])
print(f"Correlación de Pearson entre {input_features[0]} y {target_feature}: {corr:.4f}")

### 3.3 Análisis exploratorio inicial

In [ ]:
feature_name = input_features[0]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

df[feature_name].plot(kind="hist", bins=20, ax=axes[0], title=f"Distribución: {feature_name}")
axes[0].set_xlabel(feature_name)

df[target_feature].plot(kind="hist", bins=20, ax=axes[1], title=f"Distribución: {target_feature}")
axes[1].set_xlabel(target_feature)

axes[2].scatter(df[feature_name], df[target_feature], alpha=0.7)
axes[2].set_title(f"{feature_name} vs {target_feature}")
axes[2].set_xlabel(feature_name)
axes[2].set_ylabel(target_feature)

plt.tight_layout()
plt.show()

## 4. Entrenamiento del modelo de regresión linealión cruzada.

### 4.1 División de datos

In [ ]:
X = df[input_features]
y = df[target_feature]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Tamaño conjunto de entrenamiento:", X_train.shape[0])
print("Tamaño conjunto de prueba (validación):", X_test.shape[0])

### 4.2 Entrenamiento del modelo

In [ ]:
# Selección del orden del polinomio
regression_degree = 1 # <-- Modifique

# Pipeline: características polinómicas + regresión lineal
model = make_pipeline(
    PolynomialFeatures(degree=orden, include_bias=False),
    LinearRegression()
)

# Entrenamiento
model.fit(X_train, y_train)

### 4.3 Predicción

In [ ]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

### 4.4 Evaluación/validación del modelo

In [ ]:
rmse_train = mean_squared_error(y_train, y_train_pred, squared=False)
r2_train = r2_score(y_train, y_train_pred)

rmse_test = mean_squared_error(y_test, y_test_pred, squared=False)
r2_test = r2_score(y_test, y_test_pred)

print("Métricas en entrenamiento")
print(f"RMSE: {rmse_train:.4f}")
print(f"R²  : {r2_train:.4f}")
print()
print("Métricas en prueba")
print(f"RMSE: {rmse_test:.4f}")
print(f"R²  : {r2_test:.4f}")

### 4.5 Parámetros del modelo

In [ ]:
print("Modelo ajustado:")
print(f"Orden de la regresión: {regression_degree}")

poly = model.named_steps["polynomialfeatures"]
linear_model = model.named_steps["linearregression"]

term_names = poly.get_feature_names_out(input_features)
coefs = linear_model.coef_
intercept = linear_model.intercept_

terminos = [f"({coef:.4f}) * {name}" for coef, name in zip(coefs, term_names)]
ecuacion = f"{target_feature} = {intercept:.4f}"
if terminos:
    ecuacion += " + " + " + ".join(terminos)

print(ecuacion)

## 5. Visualización del ajuste

In [ ]:
feature_name = input_features[0]

x_line_values = np.linspace(X[feature_name].min(), X[feature_name].max(), 200)
x_line = pd.DataFrame({feature_name: x_line_values})
y_line = model.predict(x_line)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_train[feature_name], y_train, alpha=0.6, label="Entrenamiento")
axes[0].scatter(X_test[feature_name], y_test, alpha=0.8, label="Prueba")
axes[0].plot(x_line_values, y_line, linewidth=2, label=f"Ajuste de orden {regression_degree}")
axes[0].set_title(f"Ajuste del modelo (orden {regression_degree})")
axes[0].set_xlabel(feature_name)
axes[0].set_ylabel(target_feature)
axes[0].legend()

axes[1].scatter(y_test, y_test_pred, alpha=0.8)
min_val = min(y_test.min(), y_test_pred.min())
max_val = max(y_test.max(), y_test_pred.max())
axes[1].plot([min_val, max_val], [min_val, max_val], linewidth=2)
axes[1].set_title("Valores reales vs predichos")
axes[1].set_xlabel("Valores reales")
axes[1].set_ylabel("Valores predichos")

plt.tight_layout()
plt.show()

## Respuesta a la pregunta

Podemos usar la longitud total para predecir la longitud de la cabeza de una zarigüeya, siempre que el valor de la **RMSE** obtenido sea suficientemente bajo. La RMSE representa el error promedio de predicción en las mismas unidades de la variable de salida, por lo que permite juzgar de forma directa qué tan cerca están las predicciones de los valores reales. Si la RMSE es pequeña en comparación con la variación natural de la longitud de la cabeza, entonces el modelo tiene una capacidad predictiva aceptable. En cambio, si la RMSE es alta, la predicción no sería lo bastante precisa para afirmar que la longitud total determina adecuadamente la longitud de la cabeza.

---

### Nota técnica
En este notebook no se inspecciona el flujo del código mediante bloques `try/except`, para evitar saturar el código y mantener la lectura centrada en el análisis principal.